In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# 1. Device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치", device)

# 2. 토큰(단어) 사전 만들기
# 숫자 문자열 "1"~"10"만 단어를 사용함
## <PAD>: 길이가 다른 문장을 같은 길이로 맞출때 빈칸처럼 채우는 토큰
## <SOS>: 디코더가 출력을 시작할 때 넣는 시작 토큰
## <EOS>: 문장을 나타내는 토큰

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2

token_to_idx  = {
    "<PAD>":0,
    "<SOS>":1,
    "<EOS>":2,
    "1":3,
    "2":4,
    "3":5,
    "4":6,
    "5":7,
    "6":8,
    "7":9,
    "8":10,
    "9":11,
    "10":12
}
# index -> token으로 다시 바뀌기 위한 사전
# -> 나중에 예측 결과를 사람이 읽을 수 있게 바꾸는데 사용
idx_to_token = {v:k for k,v in token_to_idx.items()}


#전체 단어의 수
vocab_size = len(idx_to_token)

#Seq2Seq가 학습할 입력-정답 한쌍을 랜덤으로 만드는 함수
#랜덤시퀀스를 생성해서 문자열로 변환했다가 다시 숫자로 전환하는
#인코딩 디코딩을 한번 갔다오는 기본구조
def generate_sample(min_len=3, max_len=5):
    # 시퀀스 길이를 3~5 사이에서 랜덤 선택
    length = random.randint(min_len,max_len)

    # 길이 만큼 1~10 까지의 랜덤 숫자 생성
    seq = [str(random.randint(1,10)) for _ in range(length)]

    # 문자열 토큰을 숫자 인덱스 변환
    src = [token_to_idx[token] for token in seq]

    # 정답(target)은 뒤집은 시퀀스
    tgt = [token_to_idx[token] for token in reversed(seq)]

    return src,tgt
    
# 작은 샘플보기
import random

random.seed(42)
src,tgt = generate_sample(min_len=3, max_len=5)

print("src(인덱스):", src)
print("tgt(인덱스):", tgt)
print("src(토큰)", [idx_to_token[x] for x in src])
print("tgt(토큰)", [idx_to_token[x] for x in tgt]) 
 
def bulid_dataset(n_samples=1000):
    # 샘플 여러 개를 모아서 데이터셋 생성
    data = []
    for _ in range(n_samples):
        src,tgt = generate_sample()
        data.append((src,tgt))
    return data

# 학습용 데이터와 테스트용 데이터 생성
train_data = bulid_dataset(1000)
test_data = bulid_dataset(10)

# 패딩 함수와 배치 생성 함수

# RNN/GRU 여러 문장을 한꺼번에 넣을려면 고정된 길이로 맞춰야함.
# ex) [1,2,3]
##ex) [4,5] -> [4,5,PAD] / 가장 긴 것을 기준으로 PAD를 하다보면 의미가 깨질수도 있고
## 길이가 길어지다 보니 기억력이 떨어지는 경우도 보임.

def pad_sequence(seq, max_len,pad_value=PAD_IDX):
    # 현재 시퀀스 길이가 max_len보다 작으면
    ## 뒤에 PAD 토큰을 붙여서 길이를 맞춰줌.
    return seq + [pad_value] * (max_len - len(seq))
    
def make_batch(data_batch):
    # 한 배치 안의 src, decoder 입력, target을 담을 리스트
    src_batch = []
    dec_input_batch = []
    target_batch = []

    for src,tgt in data_batch:
        src_seq = src + [EOS_IDX]

        # decoder 입력은 항상 SOS로 시작-> 그 뒤에 정답 시퀀스를 붙임.
        dec_input_seq = [SOS_IDX] + tgt

        # decoder가 맞춰야 할 target은 실제 정답 EOS를 붙인 형태
        target_seq = tgt + [EOS_IDX]
        
        src_batch.append(src_seq)
        dec_input_batch.append(dec_input_seq)
        target_batch.append(target_seq)
        
    # 각 배치에서 가장 긴 길이를 찾음.
    src_max_len = max(len(x) for x in src_batch)
    dec_max_len = max(len(x) for x in dec_input_batch)
    tgt_max_len = max(len(x) for x in target_batch)
    
    # 가장 긴 길이에 맞춰 PAD 추가
    src_batch = [pad_sequence(x,src_max_len) for x in src_batch]
    dec_input_batch = [pad_sequence(x,dec_max_len) for x in dec_input_batch]
    target_batch = [pad_sequence(x,tgt_max_len) for x in target_batch]

    # 파이토치 텐서로 변환 후 device로 이동
    src_tensor = torch.tensor(src_batch, dtype=torch.long, device=device)
    dec_input_tensor = torch.tensor(dec_input_batch, dtype=torch.long, device=device)
    target_tensor = torch.tensor(target_batch, dtype=torch.long, device=device)

    return src_tensor, dec_input_tensor,target_tensor


# make_batch()가 실제로 무엇을 만드는 지?
## decoder 입력과 target이 어떻게 다른지?

sample1 = ([3,4,5],[5,4,3])
sample2 = ([6,7], [7,6])

batch = [sample1,sample2]

src_tensor, dec_input_tensor, target_tensor = make_batch(batch)

print("src_tensor")
print(src_tensor)
print()

print("dec_input_tensor")
print(dec_input_tensor)
print()

print("target_tensor")
print(target_tensor)
print()

def show_tensor_as_tokens(tensor, name):
    print(f"--- {name} ---")
    for row in tensor.tolist():
        print(row, "->", [idx_to_token[x] for x in row])
    print()

show_tensor_as_tokens(src_tensor, "src_tensor")
show_tensor_as_tokens(dec_input_tensor, "dec_input_tensor")
show_tensor_as_tokens(target_tensor, "target_tensor")

    
# Encoder
## 입력 시퀀스를 읽고 마지막 hidden layer(context vector)에 입력 문장의 정보를 압축하여 담음

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()

        # 단어 인덱스 -> Dense vector(임베딩)로 바꾸는 층
        ## 3 -> [0.1, -0.2,...]
        
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = emb_dim,
            padding_idx = PAD_IDX
        )
        
        self.gru = nn.GRU(
            input_size = emb_dim,
            hidden_size = hidden_dim,
            batch_first=True)
    def forward(self,src):
        # src : (batch_size,src_len)

        #각 토큰을 임베딩 벡터로 변환. 
        ## (embedding shape: (batch_size,src_len, emb_dim)
        embedded = self.embedding(src)

        # GRU 넣어 시퀀스를 순서대로 읽음.
        ## outputs: 각 시점의 hidden_state
        ## hidden : 마지막 시점의 hidden_state

        # Output shape: (batch_size, src_len, hidden_dim)
        # hidden_shape : (1, batch_size, hidden_dim)
        outputs, hidden = self.gru(embedded)

        return outputs,hidden
# Encoder 통과 중간 보기
emb_dim = 8
hidden_dim = 16

encoder = Encoder(vocab_size, emb_dim, hidden_dim).to(device)

src_tensor, dec_input_tensor, target_tensor = make_batch(batch)

print("src_tensor shape:", src_tensor.shape)  # (batch_size, src_len)

embedded = encoder.embedding(src_tensor)
print("embedded shape:", embedded.shape)      # (batch_size, src_len, emb_dim)

outputs, hidden = encoder(src_tensor)
print("encoder outputs shape:", outputs.shape) # (batch_size, src_len, hidden_dim)
print("encoder hidden shape:", hidden.shape)   # (1, batch_size, hidden_dim) 


# Decoder
## encoder의 hidden_state(context vector)를 받아 출력 시퀀스를 한 단계식 생성
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().init()

        # decoder 입력 토큰도 임베딩 변환
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = emb_dim,
            padding_idx = PAD_IDX
        )
        #GRU 층
        self.gru = nn.GRU(
            input_size = emb_dim,
            hidden_size = hidden_dim,
            batch_first=True)
        # hidden_state를 단어 분류 점수(Logits-> 0~1로 바꾸는 함수))로 바꾸는 선형층
        # hidden_dim -> vocab_size
        self.fc = nn.Linear(hidden_dim, vocab_size)   
    
    def forward(self,dec_input, hidden):
        # dec_input shape: (batch_size, tgt_len)
        # hidden shape : (1, batch_size, hidden_dim)

        # decoder 토큰 임베딩
        ## embedded shape: (batch_size, tgt_len, emb_dim)
        embedded = self.embedding(dec_input)

        # 이전 hidden을 받아 다음 출력 계산
        # output_shape: (batch_size, tgt_len, hidden_dim)
        outputs, hidden = self.gru(embedded, hidden)

        #각 시점마다 어떤 단어가 나와야 하는지 점수 계산
        # logits shape: (batch_size, tgt_len, vocab_size)
        logits = self.fc(outputs)
        
        return logits, hidden
# Decoder의 중간보기
decoder = Decoder(vocab_size, emb_dim, hidden_dim).to(device)

logits, dec_hidden = decoder(dec_input_tensor, hidden)

print("dec_input_tensor shape:", dec_input_tensor.shape)  # (batch_size, tgt_len)
print("logits shape:", logits.shape)                      # (batch_size, tgt_len, vocab_size)
print("dec_hidden shape:", dec_hidden.shape)              # (1, batch_size, hidden_dim)  

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self,encoder,decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self,src, dec_input):
        # 1. encoder기 입력 문장을 읽음
        _, hidden = self.encoder(src)

        # 2. encoder의 마지막 hidden을 decoder 초기 hidden에 전달
        logits, _ = self.decoder(dec_input,hidden)
        return logits